# DemandPulse

## Probabilistic Demand Forecasting & Calibration

**Notebook 02 — Quantile Forecasting, Conformal Calibration and Demand Uncertainty**

This notebook continues the methodology of `01_demandpulse_data_and_forecasting.ipynb` but is deliberately **kernel-independent**: it reloads the remote source, reconstructs deterministic series sampling, rebuilds leakage-safe features, and does not depend on Notebook 01 Python variables.

### Research question

> **How uncertain is future demand, and can statistically useful prediction intervals support later inventory-risk decisions?**

The notebook estimates horizon-specific conditional quantiles `P10`, `P50`, and `P90`, evaluates calibration and sharpness under rolling-origin validation, conformalizes the central 80% interval on a chronological **training-only** calibration window, and evaluates the frozen procedure on the official Hugging Face `eval` split.

This is a forecasting and uncertainty notebook only. It does **not** perform causal promotion analysis, inventory optimization, application development, deployment, monitoring, or agent/LLM work.


## 1. Environment


In [1]:
# %pip install -q "datasets>=3.0" "pandas>=2.0" "numpy>=1.24" \
    "pyarrow>=14" "matplotlib>=3.7" "scikit-learn>=1.3" \
    "lightgbm>=4.0" "tqdm>=4.66"


IndentationError: unexpected indent (2173150600.py, line 2)

In [ ]:
import gc
import hashlib
import json
import os
import random
import sys
from pathlib import Path
from typing import Dict, List, Mapping, Optional, Sequence, Tuple

import datasets
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from datasets import load_dataset
from IPython.display import Markdown, display
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_pinball_loss, mean_squared_error
from tqdm.auto import tqdm

RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Python      : {sys.version.split()[0]}")
print(f"datasets    : {datasets.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"LightGBM    : {lgb.__version__}")


## 2. Project configuration


In [ ]:
DATASET_ID = "Dingdong-Inc/FreshRetailNet-50K"
RANDOM_STATE = 42
SERIES_SAMPLE_PERCENT = 30
FORECAST_HORIZON = 7
QUANTILES = [0.10, 0.50, 0.90]
INTERVAL_ALPHA = 0.20
N_BACKTEST_FOLDS = 3
CALIBRATION_DAYS = 21
STREAM_BATCH_SIZE = 50_000
RUN_FULL_BACKTEST = True

assert 0 < SERIES_SAMPLE_PERCENT <= 100
assert FORECAST_HORIZON >= 1
assert QUANTILES == sorted(QUANTILES)
assert np.isclose(QUANTILES[0], INTERVAL_ALPHA / 2)
assert np.isclose(QUANTILES[-1], 1 - INTERVAL_ALPHA / 2)
assert N_BACKTEST_FOLDS >= 1
assert CALIBRATION_DAYS >= FORECAST_HORIZON

REPORTS_DIR = Path("reports")
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = Path("models")
QUANTILE_MODELS_DIR = MODELS_DIR / "quantile"
for directory in (REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, QUANTILE_MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Dataset                 : {DATASET_ID}")
print(f"Complete-series sample  : {SERIES_SAMPLE_PERCENT}%")
print(f"Forecast horizon        : {FORECAST_HORIZON} days")
print(f"Quantiles               : {QUANTILES}")
print(f"Nominal P10-P90 coverage: {1 - INTERVAL_ALPHA:.0%}")
print(f"Backtest folds          : {N_BACKTEST_FOLDS}")
print(f"Calibration window      : {CALIBRATION_DAYS} target days")
print(f"Full backtest           : {RUN_FULL_BACKTEST}")


## 3. Notebook 01 artifact check

Notebook 02 can use Notebook 01 output tables as **context**, but never assumes that its Python state is available. Missing prior artifacts generate a warning rather than blocking probabilistic modeling. The official Notebook 02 comparison uses common-row predictions produced in this notebook; Notebook 01 aggregate metrics are not treated as common-row evidence unless equivalence can actually be established.


In [ ]:
NOTEBOOK01_ARTIFACTS = {
    "baseline_backtest_results": TABLES_DIR / "baseline_backtest_results.csv",
    "final_model_comparison": TABLES_DIR / "final_model_comparison.csv",
    "segment_error_analysis": TABLES_DIR / "segment_error_analysis.csv",
    "feature_importance": TABLES_DIR / "feature_importance.csv",
    "point_model": MODELS_DIR / "demandpulse_lgbm.txt",
}

notebook01_tables: Dict[str, pd.DataFrame] = {}
for name, path in NOTEBOOK01_ARTIFACTS.items():
    if path.exists():
        if path.suffix.lower() == ".csv":
            notebook01_tables[name] = pd.read_csv(path)
        print(f"FOUND   {path}")
    else:
        print(f"WARNING missing Notebook 01 artifact: {path}")

if "final_model_comparison" in notebook01_tables:
    print("\nNotebook 01 final comparison is available for contextual inspection:")
    display(notebook01_tables["final_model_comparison"].head(20))
else:
    print("Notebook 01 final metrics are unavailable; no previous results will be fabricated.")


## 4. Remote data source and schema validation


In [ ]:
REQUIRED_COLUMNS = [
    "city_id", "store_id", "management_group_id", "first_category_id",
    "second_category_id", "third_category_id", "product_id", "dt",
    "sale_amount", "hours_sale", "stock_hour6_22_cnt", "hours_stock_status",
    "discount", "holiday_flag", "activity_flag", "precpt",
    "avg_temperature", "avg_humidity", "avg_wind_level",
]

# Same-day sequence fields are validated in the remote schema but intentionally not materialized.
MATERIALIZED_COLUMNS = [
    c for c in REQUIRED_COLUMNS if c not in {"hours_sale", "hours_stock_status"}
]
SERIES_KEYS = ["store_id", "product_id"]
ROW_KEY = SERIES_KEYS + ["dt"]
ENTITY_COLUMNS = [
    "city_id", "store_id", "management_group_id", "first_category_id",
    "second_category_id", "third_category_id", "product_id",
]
WEATHER_COLUMNS = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]

train_stream = load_dataset(DATASET_ID, split="train", streaming=True)
eval_stream = load_dataset(DATASET_ID, split="eval", streaming=True)


def stream_schema_columns(stream) -> List[str]:
    """Return schema names without materializing a streaming dataset."""
    if getattr(stream, "features", None) is not None:
        return list(stream.features.keys())
    if getattr(stream, "column_names", None) is not None:
        return list(stream.column_names)
    raise RuntimeError("Could not read streaming schema before materialization.")

train_schema = stream_schema_columns(train_stream)
eval_schema = stream_schema_columns(eval_stream)
for split_name, schema in (("train", train_schema), ("eval", eval_schema)):
    missing = sorted(set(REQUIRED_COLUMNS) - set(schema))
    if missing:
        raise AssertionError(f"{split_name} stream is missing required columns: {missing}")
    print(f"{split_name} schema verified: {len(schema)} columns")

print(f"train type: {type(train_stream).__name__}")
print(f"eval type : {type(eval_stream).__name__}")


## 5. Deterministic complete-series sampling and safe materialization

The sampling rule is intentionally identical to Notebook 01. A series is `(store_id, product_id)`. The UTF-8 token `store_id-product_id` is SHA-256 hashed; the first eight digest bytes are mapped to one of 10,000 buckets. A 30% experiment keeps buckets `< 3000`. The same rule is applied independently to train and eval, so retained entities are deterministic and sampling never takes the first fraction of rows.


In [ ]:
def series_selected(
    store_id: object,
    product_id: object,
    percent: float,
    cache: Dict[Tuple[object, object], bool],
) -> bool:
    """Notebook-01-compatible stable series selection with 0.01 percentage-point resolution."""
    if percent >= 100:
        return True
    cache_key = (store_id, product_id)
    if cache_key not in cache:
        token = f"{store_id}-{product_id}".encode("utf-8")
        digest = hashlib.sha256(token).digest()
        bucket = int.from_bytes(digest[:8], byteorder="big", signed=False) % 10_000
        cache[cache_key] = bucket < int(round(percent * 100))
    return cache[cache_key]


def materialize_stream(stream, split_name, sample_percent, batch_size=50_000):
    """
    Materialize a deterministic subset of complete store-product series.
    """
    import time
    if not hasattr(stream, "batch"):
        raise RuntimeError(
            "This notebook expects a recent Hugging Face datasets version with IterableDataset.batch(). "
            "Re-run the install cell and restart the kernel if necessary."
        )

    frames = []
    selection_cache = {}
    selected_missing_counts = {c: 0 for c in REQUIRED_COLUMNS}
    rows_processed = 0
    rows_selected = 0

    max_retries = 5
    for attempt in range(max_retries):
        try:
            frames.clear()
            rows_processed = 0
            rows_selected = 0
            selected_missing_counts = {c: 0 for c in REQUIRED_COLUMNS}
            batched_stream = stream.batch(batch_size=batch_size)
            for batch in tqdm(batched_stream, desc=f"Streaming {split_name}", unit="batch"):
                batch_df = pd.DataFrame(batch)
                rows_processed += len(batch_df)

                keep_mask = np.fromiter(
                    (
                        series_selected(store_id, product_id, sample_percent, selection_cache)
                        for store_id, product_id in zip(batch_df["store_id"], batch_df["product_id"])
                    ),
                    dtype=bool,
                    count=len(batch_df),
                )
                if not keep_mask.any():
                    continue

                selected = batch_df.loc[keep_mask]
                rows_selected += len(selected)
                for column in REQUIRED_COLUMNS:
                    selected_missing_counts[column] += int(selected[column].isna().sum())

                frames.append(selected[MATERIALIZED_COLUMNS].copy())
            break
        except Exception as err:
            if attempt == max_retries - 1:
                raise err
            print(f"Network glitch during {split_name} streaming (attempt {attempt + 1}/{max_retries}): {err}. Retrying in 3s...")
            time.sleep(3)

    if not frames:
        raise RuntimeError(
            f"No rows were selected from {split_name}. Check SERIES_SAMPLE_PERCENT and source availability."
        )

    result = pd.concat(frames, ignore_index=True, copy=False)
    result.attrs["rows_processed"] = rows_processed
    result.attrs["rows_selected"] = rows_selected
    result.attrs["selected_missing_counts"] = selected_missing_counts
    result.attrs["schema_types"] = train_schema_types if split_name == "train" else eval_schema_types

    print(
        f"{split_name}: processed {rows_processed:,} rows; "
        f"selected {rows_selected:,} rows ({rows_selected / rows_processed:.1%})."
    )
    return result

In [ ]:
train_df = materialize_stream(train_stream, "train", SERIES_SAMPLE_PERCENT)
eval_df = materialize_stream(eval_stream, "eval", SERIES_SAMPLE_PERCENT)

train_audit = dict(train_df.attrs)
eval_audit = dict(eval_df.attrs)

train_df["dt"] = pd.to_datetime(train_df["dt"], errors="raise")
eval_df["dt"] = pd.to_datetime(eval_df["dt"], errors="raise")

print(f"train shape: {train_df.shape}; memory={memory_mb(train_df):,.1f} MB")
print(f"eval shape : {eval_df.shape}; memory={memory_mb(eval_df):,.1f} MB")
print(f"train dates: {train_df['dt'].min().date()} -> {train_df['dt'].max().date()}")
print(f"eval dates : {eval_df['dt'].min().date()} -> {eval_df['dt'].max().date()}")


## 6. Data validation and cold-start classification


In [ ]:
# Materialized schema
missing_materialized_train = sorted(set(MATERIALIZED_COLUMNS) - set(train_df.columns))
missing_materialized_eval = sorted(set(MATERIALIZED_COLUMNS) - set(eval_df.columns))
assert not missing_materialized_train, f"train materialization missing: {missing_materialized_train}"
assert not missing_materialized_eval, f"eval materialization missing: {missing_materialized_eval}"

# Duplicate key integrity: display before stopping.
for split_name, frame in (("train", train_df), ("eval", eval_df)):
    duplicate_mask = frame.duplicated(ROW_KEY, keep=False)
    n_duplicate_rows = int(duplicate_mask.sum())
    print(f"{split_name} duplicate key rows: {n_duplicate_rows:,}")
    if n_duplicate_rows:
        display(frame.loc[duplicate_mask].sort_values(ROW_KEY).head(50))
        raise ValueError(f"Duplicate {ROW_KEY} rows exist in {split_name}; not removed automatically.")

# Nonnegative, observed target.
for split_name, frame in (("train", train_df), ("eval", eval_df)):
    n_missing_target = int(frame["sale_amount"].isna().sum())
    n_negative_target = int((frame["sale_amount"] < 0).sum())
    print(f"{split_name}: missing target={n_missing_target:,}, negative target={n_negative_target:,}")
    assert n_missing_target == 0, f"Missing sale_amount in {split_name}"
    assert n_negative_target == 0, f"Negative sale_amount in {split_name}"

train_min, train_max = train_df["dt"].min(), train_df["dt"].max()
eval_min, eval_max = eval_df["dt"].min(), eval_df["dt"].max()
HOLDOUT_TEMPORALLY_VALID = bool(train_max < eval_min)
print(f"train: {train_min.date()} -> {train_max.date()}")
print(f"eval : {eval_min.date()} -> {eval_max.date()}")
print(f"max(train) < min(eval): {HOLDOUT_TEMPORALLY_VALID}")
if not HOLDOUT_TEMPORALLY_VALID:
    print("WARNING: official holdout is not strictly after training. Backtesting may continue, but final holdout evaluation will be stopped.")

train_series_df = train_df[SERIES_KEYS].drop_duplicates()
eval_series_df = eval_df[SERIES_KEYS].drop_duplicates()
train_series = set(map(tuple, train_series_df.to_numpy()))
eval_series = set(map(tuple, eval_series_df.to_numpy()))
existing_series = train_series & eval_series
cold_start_series = eval_series - train_series
train_only_series = train_series - eval_series

print(f"rows processed train : {train_audit['rows_processed']:,}")
print(f"rows selected train  : {train_audit['rows_selected']:,}")
print(f"rows processed eval  : {eval_audit['rows_processed']:,}")
print(f"rows selected eval   : {eval_audit['rows_selected']:,}")
print(f"train series         : {len(train_series):,}")
print(f"eval series          : {len(eval_series):,}")
print(f"series present both  : {len(existing_series):,}")
print(f"train-only series    : {len(train_only_series):,}")
print(f"cold-start series    : {len(cold_start_series):,}")

status_records = []
for key in sorted(train_series | eval_series):
    if key in existing_series:
        status = "existing_series"
    elif key in cold_start_series:
        status = "cold_start_series"
    else:
        status = "train_only_series"
    status_records.append({"store_id": key[0], "product_id": key[1], "series_status": status})
series_status_df = pd.DataFrame(status_records)
display(series_status_df["series_status"].value_counts().rename_axis("series_status").reset_index(name="series_count"))


## 7. Probabilistic forecasting concept

A point forecast summarizes one location of the conditional outcome distribution, often the conditional mean or median depending on the objective. Probabilistic forecasting instead estimates characteristics of the conditional distribution `P(Y | X)`.

For quantile level `q`, the target is the conditional quantile `Q_q(Y | X)`. Here three separate LightGBM models estimate `Q_0.10`, `Q_0.50`, and `Q_0.90` for each forecast horizon. These are **conditional quantile estimates, not Bayesian posterior distributions**. The P10–P90 span is a nominal central 80% prediction interval.

### Coverage versus sharpness

Coverage alone is not sufficient: a very wide interval can cover almost everything while being operationally uninformative. We therefore evaluate empirical coverage together with width and interval score. Useful intervals aim for appropriate coverage **and** reasonable sharpness.


## 8. Pinball loss and metric validation


In [ ]:
def pinball_loss(y_true: Sequence[float], y_pred: Sequence[float], quantile: float) -> float:
    """Mean quantile (pinball) loss, implemented directly from its piecewise definition."""
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    if y.shape != p.shape:
        raise ValueError(f"Shape mismatch: y_true={y.shape}, y_pred={p.shape}")
    if not 0 < quantile < 1:
        raise ValueError("quantile must be strictly between 0 and 1")
    e = y - p
    loss = np.where(e >= 0, quantile * e, (quantile - 1.0) * e)
    return float(np.mean(loss))

synthetic_y = np.array([0.0, 1.0, 3.0, 5.0, 8.0])
synthetic_p = np.array([0.5, 1.5, 2.0, 6.0, 7.0])
for q in QUANTILES:
    manual = pinball_loss(synthetic_y, synthetic_p, q)
    sklearn_value = mean_pinball_loss(synthetic_y, synthetic_p, alpha=q)
    print(f"q={q:.2f}: manual={manual:.12f}, sklearn={sklearn_value:.12f}")
    assert np.isclose(manual, sklearn_value, rtol=1e-12, atol=1e-12)
print("Pinball-loss implementation verified against scikit-learn.")


# Leakage Audit

**Target:** `sale_amount`.

Same-day `hours_sale`, `hours_stock_status`, and `stock_hour6_22_cnt` are never used as predictors for the date being forecast. The two sequence fields are not materialized at all. `stock_hour6_22_cnt` is retained only to construct **historical** stockout information.

All sales lags and rolling features are shifted before aggregation. For an origin row dated `t`, `lag_1` is demand from `t-1`, and rolling windows end at `t-1`. This deliberately preserves Notebook 01's conservative shifted-feature convention. Future target `sale_amount` is used only as a label after an exact date-aligned merge.

For each horizon, training examples are filtered by **target date**, not origin date alone: a fold with cutoff `T` may only fit examples whose `target_date <= T`. Validation target dates are strictly after the fold origin. The official `eval` split is not used for model selection or conformal calibration.


## 9. Direct multi-horizon forecasting

Notebook 02 uses a direct strategy:

- `Q(Y[t+1] | X[t])`
- `Q(Y[t+2] | X[t])`
- …
- `Q(Y[t+7] | X[t])`

Each horizon receives a separate P10, P50, and P90 model, producing **21 core quantile models**. This avoids recursively feeding predicted demand into future lag features, which would complicate uncertainty propagation.

**Advantages:** horizon-specific behavior, direct uncertainty interpretation, no recursive target replacement.  
**Disadvantages:** more models, higher training cost, and no guarantee that forecasts across horizons form mutually coherent sample paths.

Realized target-date weather is excluded from core models. Holiday, discount, and activity features are treated as future-known only under the explicit assumption that schedules/plans are available before prediction time.


## 10. Origin-safe feature engineering and supervised horizon builder


In [ ]:
HISTORY_FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_28",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28",
    "rolling_min_7", "rolling_max_7", "stockout_lag_7",
]
FORECAST_CALENDAR_FEATURES = [
    "forecast_day_of_week", "forecast_day_of_month", "forecast_week_of_year",
    "forecast_month", "forecast_is_weekend",
]
FUTURE_KNOWN_FEATURES = [
    "future_discount", "future_discount_pct", "future_holiday_flag", "future_activity_flag",
]
CORE_FEATURE_COLUMNS = ENTITY_COLUMNS + HISTORY_FEATURES + FORECAST_CALENDAR_FEATURES + FUTURE_KNOWN_FEATURES
CATEGORICAL_FEATURES = ENTITY_COLUMNS.copy()
MODEL_HISTORY_REQUIREMENTS = ["lag_28", "rolling_mean_28", "rolling_std_28"]


def build_origin_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create origin-date features using target history strictly before each origin date."""
    out = df.sort_values(SERIES_KEYS + ["dt"]).copy()
    out["stockout_flag"] = out["stock_hour6_22_cnt"].gt(0).astype("int8")
    grouped = out.groupby(SERIES_KEYS, sort=False, observed=True)

    for lag in (1, 2, 3, 7, 14, 28):
        out[f"lag_{lag}"] = grouped["sale_amount"].shift(lag)

    for window in (7, 14, 28):
        out[f"rolling_mean_{window}"] = grouped["sale_amount"].transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=w).mean()
        )
        out[f"rolling_std_{window}"] = grouped["sale_amount"].transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=w).std()
        )

    out["rolling_min_7"] = grouped["sale_amount"].transform(
        lambda s: s.shift(1).rolling(7, min_periods=7).min()
    )
    out["rolling_max_7"] = grouped["sale_amount"].transform(
        lambda s: s.shift(1).rolling(7, min_periods=7).max()
    )
    out["stockout_lag_7"] = grouped["stockout_flag"].shift(7)
    out = out.rename(columns={"dt": "origin_date"})
    return out


def _target_lookup(df: pd.DataFrame) -> pd.DataFrame:
    """Target-date labels and production-known target-date covariates."""
    lookup = df[[
        *SERIES_KEYS, "dt", "sale_amount", "discount", "holiday_flag", "activity_flag",
        *WEATHER_COLUMNS,
    ]].copy()
    lookup = lookup.rename(columns={
        "dt": "target_date",
        "sale_amount": "target",
        "discount": "future_discount",
        "holiday_flag": "future_holiday_flag",
        "activity_flag": "future_activity_flag",
        "precpt": "future_precpt",
        "avg_temperature": "future_avg_temperature",
        "avg_humidity": "future_avg_humidity",
        "avg_wind_level": "future_avg_wind_level",
    })
    lookup["future_discount_pct"] = 1.0 - lookup["future_discount"]
    return lookup


def build_horizon_dataset(
    origin_features: pd.DataFrame,
    source_df: pd.DataFrame,
    horizon: int,
    max_target_date: Optional[pd.Timestamp] = None,
    min_target_date: Optional[pd.Timestamp] = None,
) -> pd.DataFrame:
    """Build exact calendar-day direct-horizon examples without row-shift assumptions."""
    if horizon < 1:
        raise ValueError("horizon must be >= 1")

    base_cols = [*ENTITY_COLUMNS, "origin_date", *HISTORY_FEATURES]
    base = origin_features[base_cols].copy()
    base["target_date"] = base["origin_date"] + pd.to_timedelta(horizon, unit="D")
    if max_target_date is not None:
        base = base.loc[base["target_date"] <= pd.Timestamp(max_target_date)]
    if min_target_date is not None:
        base = base.loc[base["target_date"] >= pd.Timestamp(min_target_date)]

    lookup = _target_lookup(source_df)
    out = base.merge(lookup, on=[*SERIES_KEYS, "target_date"], how="inner", validate="one_to_one")
    out["horizon"] = int(horizon)
    out["forecast_day_of_week"] = out["target_date"].dt.dayofweek.astype("int8")
    out["forecast_day_of_month"] = out["target_date"].dt.day.astype("int8")
    out["forecast_week_of_year"] = out["target_date"].dt.isocalendar().week.astype("int16")
    out["forecast_month"] = out["target_date"].dt.month.astype("int8")
    out["forecast_is_weekend"] = out["forecast_day_of_week"].isin([5, 6]).astype("int8")

    assert (out["target_date"] > out["origin_date"]).all(), "target_date must be after origin_date"
    expected_h = (out["target_date"] - out["origin_date"]).dt.days
    assert (expected_h == horizon).all(), "Direct-horizon date alignment failed"
    if max_target_date is not None:
        assert out["target_date"].max() <= pd.Timestamp(max_target_date)
    return out


train_origin_features = build_origin_features(train_df)
print(f"Origin feature rows: {len(train_origin_features):,}")
print("Missing model-history requirements:")
display(train_origin_features[MODEL_HISTORY_REQUIREMENTS].isna().sum().rename("missing").reset_index(name="feature"))


### Feature validation

The earliest rows of each series necessarily lack long lags and 28-day rolling statistics. For model fitting, rows missing the explicitly required 28-day history are removed; LightGBM may still handle occasional missing values in other predictors. This is a history-availability rule, not arbitrary row deletion.


In [ ]:
def validate_feature_frame(frame: pd.DataFrame, feature_columns: Sequence[str]) -> None:
    """Fail on duplicate supervised keys or infinite numeric feature values."""
    key = [*SERIES_KEYS, "origin_date", "target_date", "horizon"]
    dup = frame.duplicated(key, keep=False)
    if dup.any():
        display(frame.loc[dup].sort_values(key).head(50))
        raise ValueError("Duplicate supervised horizon rows detected.")

    numeric = frame[list(feature_columns)].apply(pd.to_numeric, errors="coerce")
    inf = np.isinf(numeric.to_numpy(dtype=float, na_value=np.nan))
    if inf.any():
        raise ValueError(f"Infinite feature values detected: {int(inf.sum())}")


def history_ready(frame: pd.DataFrame) -> pd.DataFrame:
    ready = frame.dropna(subset=MODEL_HISTORY_REQUIREMENTS).copy()
    return ready

sample_horizon_check = build_horizon_dataset(
    train_origin_features,
    train_df,
    horizon=1,
    max_target_date=train_max,
)
validate_feature_frame(sample_horizon_check, CORE_FEATURE_COLUMNS)
print(f"H1 supervised rows before history rule: {len(sample_horizon_check):,}")
print(f"H1 rows after 28-day history rule       : {len(history_ready(sample_horizon_check)):,}")
del sample_horizon_check
gc.collect()


## 11. Safe categorical handling


In [ ]:
def fit_category_maps(frame: pd.DataFrame, columns: Sequence[str]) -> Dict[str, List[object]]:
    """Store training-only category vocabularies for safe LightGBM integer coding."""
    maps: Dict[str, List[object]] = {}
    for col in columns:
        values = pd.Series(frame[col].dropna().unique()).sort_values().tolist()
        maps[col] = values
    return maps


def apply_category_maps(
    frame: pd.DataFrame,
    maps: Mapping[str, Sequence[object]],
    columns: Sequence[str],
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """Map known categories to nonnegative integer codes and unseen values to -1 (LightGBM missing category)."""
    out = frame.copy()
    unknown_counts: Dict[str, int] = {}
    for col in columns:
        vocab = list(maps[col])
        mapping = {value: idx for idx, value in enumerate(vocab)}
        encoded = out[col].map(mapping)
        unknown_counts[col] = int(encoded.isna().sum())
        out[col] = encoded.fillna(-1).astype("int32")
    return out, unknown_counts


def model_matrix(
    frame: pd.DataFrame,
    category_maps: Mapping[str, Sequence[object]],
    feature_columns: Sequence[str] = CORE_FEATURE_COLUMNS,
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    encoded, unknown = apply_category_maps(frame[list(feature_columns)], category_maps, CATEGORICAL_FEATURES)
    return encoded, unknown


## 12. Probabilistic, interval, and point metrics


In [ ]:
def safe_wape(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    denom = np.abs(y).sum()
    return float(np.abs(y - p).sum() / denom) if denom > 0 else np.nan


def safe_bias(y_true: Sequence[float], y_pred: Sequence[float]) -> float:
    """Positive bias means aggregate over-forecasting; negative means under-forecasting."""
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    denom = np.abs(y).sum()
    return float((p - y).sum() / denom) if denom > 0 else np.nan


def calculate_point_metrics(y_true: Sequence[float], y_pred: Sequence[float]) -> Dict[str, float]:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p)
    y, p = y[mask], p[mask]
    if len(y) == 0:
        return {"mae": np.nan, "rmse": np.nan, "wape": np.nan, "bias": np.nan, "observations": 0}
    return {
        "mae": float(mean_absolute_error(y, p)),
        "rmse": float(mean_squared_error(y, p) ** 0.5),
        "wape": safe_wape(y, p),
        "bias": safe_bias(y, p),
        "observations": int(len(y)),
    }


def interval_score(
    y_true: Sequence[float], lower: Sequence[float], upper: Sequence[float], alpha: float = INTERVAL_ALPHA
) -> float:
    """Mean standard central prediction-interval score; lower is better."""
    y = np.asarray(y_true, dtype=float)
    lo = np.asarray(lower, dtype=float)
    hi = np.asarray(upper, dtype=float)
    width = hi - lo
    score = width.copy()
    score += (2.0 / alpha) * (lo - y) * (y < lo)
    score += (2.0 / alpha) * (y - hi) * (y > hi)
    return float(np.mean(score))


def calculate_probabilistic_metrics(
    y_true: Sequence[float], p10: Sequence[float], p50: Sequence[float], p90: Sequence[float]
) -> Dict[str, float]:
    y = np.asarray(y_true, dtype=float)
    q10 = np.asarray(p10, dtype=float)
    q50 = np.asarray(p50, dtype=float)
    q90 = np.asarray(p90, dtype=float)
    mask = np.isfinite(y) & np.isfinite(q10) & np.isfinite(q50) & np.isfinite(q90)
    y, q10, q50, q90 = y[mask], q10[mask], q50[mask], q90[mask]
    if len(y) == 0:
        return {k: np.nan for k in [
            "p10_pinball", "p50_pinball", "p90_pinball", "mean_pinball", "coverage",
            "mean_width", "median_width", "p90_width", "interval_score"
        ]} | {"observations": 0}
    widths = q90 - q10
    losses = [pinball_loss(y, q, level) for q, level in zip((q10, q50, q90), QUANTILES)]
    return {
        "p10_pinball": losses[0],
        "p50_pinball": losses[1],
        "p90_pinball": losses[2],
        "mean_pinball": float(np.mean(losses)),
        "coverage": float(np.mean((q10 <= y) & (y <= q90))),
        "mean_width": float(np.mean(widths)),
        "median_width": float(np.median(widths)),
        "p90_width": float(np.quantile(widths, 0.90)),
        "interval_score": interval_score(y, q10, q90, INTERVAL_ALPHA),
        "observations": int(len(y)),
    }


def crossing_diagnostics(frame: pd.DataFrame, prefix: str = "raw") -> Dict[str, float]:
    p10, p50, p90 = (frame[f"{prefix}_p10"], frame[f"{prefix}_p50"], frame[f"{prefix}_p90"])
    crossed = (p10 > p50) | (p50 > p90) | (p10 > p90)
    return {
        "crossing_count": int(crossed.sum()),
        "crossing_percentage": float(crossed.mean()) if len(frame) else np.nan,
        "p10_gt_p50": int((p10 > p50).sum()),
        "p50_gt_p90": int((p50 > p90).sum()),
        "p10_gt_p90": int((p10 > p90).sum()),
        "observations": int(len(frame)),
    }


def repair_quantile_crossing(frame: pd.DataFrame, input_prefix: str, output_prefix: str) -> pd.DataFrame:
    """Monotonic repair by row-wise sorting; diagnostics must be run before this call."""
    out = frame.copy()
    values = out[[f"{input_prefix}_p10", f"{input_prefix}_p50", f"{input_prefix}_p90"]].to_numpy(dtype=float)
    repaired = np.sort(values, axis=1)
    out[f"{output_prefix}_p10"] = repaired[:, 0]
    out[f"{output_prefix}_p50"] = repaired[:, 1]
    out[f"{output_prefix}_p90"] = repaired[:, 2]
    return out


def operationalize_quantiles(frame: pd.DataFrame) -> pd.DataFrame:
    """Preserve raw predictions, measure negatives externally, then clip to nonnegative and repair crossing."""
    out = frame.copy()
    for q in (10, 50, 90):
        out[f"clipped_p{q}"] = np.maximum(out[f"raw_p{q}"].to_numpy(dtype=float), 0.0)
    out = repair_quantile_crossing(out, "clipped", "p")
    return out


## 13. Rolling-origin folds reserved before the conformal calibration window


In [ ]:
calibration_end = train_max
calibration_start = calibration_end - pd.Timedelta(days=CALIBRATION_DAYS - 1)
model_fit_end = calibration_start - pd.Timedelta(days=1)
backtest_pool_end = model_fit_end


def make_rolling_origins(
    available_dates: Sequence[pd.Timestamp],
    n_folds: int,
    horizon: int,
    pool_end: pd.Timestamp,
) -> pd.DataFrame:
    """Construct non-overlapping recent rolling origins whose validation remains before calibration."""
    dates = pd.DatetimeIndex(sorted(pd.to_datetime(pd.Series(available_dates).dropna().unique())))
    dates = dates[dates <= pd.Timestamp(pool_end)]
    if len(dates) < 28 + horizon * (n_folds + 1):
        raise RuntimeError("Insufficient pre-calibration history for the requested rolling-origin design.")

    latest_origin = pd.Timestamp(pool_end) - pd.Timedelta(days=horizon)
    records = []
    for reverse_idx in range(n_folds):
        origin = latest_origin - pd.Timedelta(days=horizon * (n_folds - 1 - reverse_idx))
        if origin not in dates:
            candidates = dates[dates <= origin]
            if len(candidates) == 0:
                raise RuntimeError("Could not align a fold origin to an available training date.")
            origin = candidates.max()
        records.append({
            "fold": reverse_idx + 1,
            "train_end": origin,
            "validation_start": origin + pd.Timedelta(days=1),
            "validation_end": origin + pd.Timedelta(days=horizon),
        })
    folds = pd.DataFrame(records)
    assert (folds["validation_end"] < calibration_start).all(), "Backtesting must not consume calibration targets."
    return folds

rolling_folds = make_rolling_origins(
    train_df["dt"], N_BACKTEST_FOLDS, FORECAST_HORIZON, backtest_pool_end
)
print(f"Model-fit period ends : {model_fit_end.date()}")
print(f"Calibration targets   : {calibration_start.date()} -> {calibration_end.date()}")
display(rolling_folds)


## 14. Quantile LightGBM training helpers


In [ ]:
LGBM_QUANTILE_PARAMS = {
    "learning_rate": 0.05,
    "n_estimators": 1000,
    "num_leaves": 63,
    "min_child_samples": 40,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}


def train_quantile_model(
    train_frame: pd.DataFrame,
    quantile: float,
    feature_columns: Sequence[str] = CORE_FEATURE_COLUMNS,
) -> Tuple[LGBMRegressor, Dict[str, List[object]]]:
    """Fit one global horizon/quantile LightGBM with training-only category vocabularies."""
    if not 0 < quantile < 1:
        raise ValueError("quantile must be in (0, 1)")
    maps = fit_category_maps(train_frame, CATEGORICAL_FEATURES)
    X, _ = model_matrix(train_frame, maps, feature_columns)
    y = train_frame["target"].to_numpy(dtype=float)
    model = LGBMRegressor(objective="quantile", alpha=quantile, **LGBM_QUANTILE_PARAMS)
    model.fit(X, y, categorical_feature=CATEGORICAL_FEATURES)
    return model, maps


def predict_quantiles(
    models: Mapping[float, LGBMRegressor],
    category_maps: Mapping[float, Mapping[str, Sequence[object]]],
    frame: pd.DataFrame,
    feature_columns: Sequence[str] = CORE_FEATURE_COLUMNS,
) -> Tuple[pd.DataFrame, Dict[float, Dict[str, int]]]:
    """Predict raw P10/P50/P90 and return unseen-category diagnostics by quantile."""
    out = frame.copy()
    unknown_by_q: Dict[float, Dict[str, int]] = {}
    for q in QUANTILES:
        X, unknown = model_matrix(frame, category_maps[q], feature_columns)
        out[f"raw_p{int(q * 100):02d}"] = models[q].predict(X)
        unknown_by_q[q] = unknown
    return out, unknown_by_q


def add_seasonal_naive(
    forecast_frame: pd.DataFrame,
    history_df: pd.DataFrame,
) -> pd.DataFrame:
    """Leakage-safe seasonal naive using demand exactly seven calendar days before each target date."""
    out = forecast_frame.copy()
    out["seasonal_reference_date"] = out["target_date"] - pd.Timedelta(days=7)
    history = history_df[[*SERIES_KEYS, "dt", "sale_amount"]].rename(
        columns={"dt": "seasonal_reference_date", "sale_amount": "seasonal_naive"}
    )
    out = out.merge(history, on=[*SERIES_KEYS, "seasonal_reference_date"], how="left", validate="many_to_one")
    return out


## 15. Full rolling-origin quantile backtest

`RUN_FULL_BACKTEST=True` executes all folds × 7 horizons × 3 quantiles. For each horizon, the supervised table is built once and discarded after that horizon to control memory. Within every fold, the model-fitting subset is filtered on `target_date <= train_end`; the validation row set is the exact fixed origin `train_end` forecasting the single corresponding horizon date.


In [ ]:
backtest_metric_records: List[Dict[str, object]] = []
crossing_records: List[Dict[str, object]] = []
seasonal_residual_frames: List[pd.DataFrame] = []
backtest_prediction_frames: List[pd.DataFrame] = []

folds_to_run = rolling_folds if RUN_FULL_BACKTEST else rolling_folds.tail(1)
if not RUN_FULL_BACKTEST:
    print("DEBUG MODE: reduced backtest is running. Do not present these results as final scientific results.")

for horizon in range(1, FORECAST_HORIZON + 1):
    print(f"\n=== Backtest horizon {horizon} ===")
    horizon_all = build_horizon_dataset(
        train_origin_features,
        train_df,
        horizon=horizon,
        max_target_date=calibration_start - pd.Timedelta(days=1),
    )
    horizon_all = history_ready(horizon_all)
    validate_feature_frame(horizon_all, CORE_FEATURE_COLUMNS)

    for fold_row in folds_to_run.itertuples(index=False):
        train_cutoff = pd.Timestamp(fold_row.train_end)
        validation_date = train_cutoff + pd.Timedelta(days=horizon)
        fit_frame = horizon_all.loc[horizon_all["target_date"] <= train_cutoff].copy()
        valid_frame = horizon_all.loc[
            (horizon_all["origin_date"] == train_cutoff)
            & (horizon_all["target_date"] == validation_date)
        ].copy()

        if fit_frame.empty or valid_frame.empty:
            raise RuntimeError(
                f"Empty fold data for fold={fold_row.fold}, horizon={horizon}; "
                "check temporal alignment and sampling."
            )
        assert fit_frame["target_date"].max() <= train_cutoff
        assert (valid_frame["target_date"] > train_cutoff).all()
        assert (valid_frame["origin_date"] == train_cutoff).all()

        print(
            f"fold={fold_row.fold} h={horizon} train_rows={len(fit_frame):,} "
            f"training_target_max={fit_frame['target_date'].max().date()} "
            f"validation_date={validation_date.date()} valid_rows={len(valid_frame):,}"
        )

        models: Dict[float, LGBMRegressor] = {}
        maps_by_q: Dict[float, Dict[str, List[object]]] = {}
        for q in QUANTILES:
            model, maps = train_quantile_model(fit_frame, q)
            models[q] = model
            maps_by_q[q] = maps

        pred, _ = predict_quantiles(models, maps_by_q, valid_frame)

        # Diagnostics BEFORE operational clipping / crossing repair.
        neg_counts = {f"negative_p{int(q*100):02d}": int((pred[f'raw_p{int(q*100):02d}'] < 0).sum()) for q in QUANTILES}
        cross = crossing_diagnostics(pred, "raw")
        crossing_records.append({"stage": "backtest", "fold": int(fold_row.fold), "horizon": horizon, **cross, **neg_counts})

        pred = operationalize_quantiles(pred)
        pred = add_seasonal_naive(pred, train_df.loc[train_df["dt"] <= train_cutoff])
        pred["fold"] = int(fold_row.fold)

        # Explicit common mask for LightGBM-vs-seasonal comparisons.
        common_evaluation_mask = pred[["target", "p_p10", "p_p50", "p_p90", "seasonal_naive"]].notna().all(axis=1)
        total_possible = len(pred)
        rows_evaluated = int(common_evaluation_mask.sum())
        rows_excluded = total_possible - rows_evaluated
        coverage_pct = rows_evaluated / total_possible if total_possible else np.nan
        compare = pred.loc[common_evaluation_mask].copy()

        prob = calculate_probabilistic_metrics(compare["target"], compare["p_p10"], compare["p_p50"], compare["p_p90"])
        point = calculate_point_metrics(compare["target"], compare["p_p50"])
        seasonal_point = calculate_point_metrics(compare["target"], compare["seasonal_naive"])

        backtest_metric_records.append({
            "fold": int(fold_row.fold), "horizon": horizon, "method": "LightGBM Quantile (operational repaired)",
            "total_possible_rows": total_possible, "rows_evaluated": rows_evaluated,
            "prediction_coverage_percentage": coverage_pct, "rows_excluded": rows_excluded,
            **point, **prob,
        })
        backtest_metric_records.append({
            "fold": int(fold_row.fold), "horizon": horizon, "method": "Seasonal Naive point",
            "total_possible_rows": total_possible, "rows_evaluated": rows_evaluated,
            "prediction_coverage_percentage": coverage_pct, "rows_excluded": rows_excluded,
            **seasonal_point,
        })

        residuals = compare[["fold", "horizon", *SERIES_KEYS, "target_date", "target", "seasonal_naive"]].copy()
        residuals["seasonal_residual"] = residuals["target"] - residuals["seasonal_naive"]
        seasonal_residual_frames.append(residuals)
        backtest_prediction_frames.append(compare[[
            "fold", "horizon", *SERIES_KEYS, "origin_date", "target_date", "target",
            "raw_p10", "raw_p50", "raw_p90", "p_p10", "p_p50", "p_p90", "seasonal_naive",
        ]].copy())

        del models, maps_by_q, pred, compare, fit_frame, valid_frame
        gc.collect()

    del horizon_all
    gc.collect()

probabilistic_backtest_results = pd.DataFrame(backtest_metric_records)
quantile_crossing_diagnostics = pd.DataFrame(crossing_records)
seasonal_backtest_residuals = pd.concat(seasonal_residual_frames, ignore_index=True)
backtest_predictions = pd.concat(backtest_prediction_frames, ignore_index=True)

probabilistic_backtest_results.to_csv(TABLES_DIR / "probabilistic_backtest_results.csv", index=False)
print(f"Saved {TABLES_DIR / 'probabilistic_backtest_results.csv'}")
display(probabilistic_backtest_results.head(20))


## 16. Probabilistic seasonal-naive residual baseline


In [ ]:
seasonal_residual_quantiles = (
    seasonal_backtest_residuals.groupby("horizon", observed=True)["seasonal_residual"]
    .quantile([0.10, 0.50, 0.90])
    .unstack()
    .rename(columns={0.10: "resid_q10", 0.50: "resid_q50", 0.90: "resid_q90"})
    .reset_index()
)
print("Residual quantiles are estimated from training-period rolling backtests only:")
display(seasonal_residual_quantiles)


# Conformal Calibration

The final `CALIBRATION_DAYS=21` of **training targets** are reserved chronologically. Model fitting ends before the calibration window; official eval is never used to estimate conformal adjustments.

For each horizon, after operational nonnegative clipping and monotonic repair, the nonconformity score is:

`max(lower_prediction - actual, actual - upper_prediction)`.

The finite-sample quantile level is `ceil((n + 1) * (1 - alpha)) / n`, capped at 1, and `np.quantile(..., method="higher")` is used when available. The conformal interval is then `[P10 - q_hat, P90 + q_hat]`, with the lower demand bound clipped at zero. P50 is **not** modified by interval calibration.


## 17. Final chronological fit + training-only conformal calibration


In [ ]:
def conformal_quantile_level(n: int, alpha: float) -> float:
    if n <= 0:
        raise ValueError("Calibration sample size must be positive")
    return float(min(1.0, np.ceil((n + 1) * (1 - alpha)) / n))


def calculate_conformal_adjustment(
    actual: Sequence[float], lower: Sequence[float], upper: Sequence[float], alpha: float = INTERVAL_ALPHA
) -> Tuple[float, float, np.ndarray]:
    y = np.asarray(actual, dtype=float)
    lo = np.asarray(lower, dtype=float)
    hi = np.asarray(upper, dtype=float)
    scores = np.maximum(lo - y, y - hi)
    level = conformal_quantile_level(len(scores), alpha)
    try:
        q_hat = float(np.quantile(scores, level, method="higher"))
    except TypeError:  # NumPy compatibility fallback; not a globally hidden warning.
        q_hat = float(np.quantile(scores, level, interpolation="higher"))
    return q_hat, level, scores


def apply_conformal_interval(
    frame: pd.DataFrame, q_hat: float, lower_col: str = "p_p10", upper_col: str = "p_p90"
) -> pd.DataFrame:
    out = frame.copy()
    out["conformal_lower"] = np.maximum(out[lower_col] - q_hat, 0.0)
    out["conformal_upper"] = out[upper_col] + q_hat
    return out

final_models: Dict[int, Dict[float, LGBMRegressor]] = {}
final_category_maps: Dict[int, Dict[float, Dict[str, List[object]]]] = {}
conformal_records: List[Dict[str, object]] = []
calibration_prediction_frames: List[pd.DataFrame] = []

for horizon in range(1, FORECAST_HORIZON + 1):
    print(f"\n=== Final fit/calibration horizon {horizon} ===")
    horizon_all = build_horizon_dataset(train_origin_features, train_df, horizon=horizon, max_target_date=train_max)
    horizon_all = history_ready(horizon_all)
    validate_feature_frame(horizon_all, CORE_FEATURE_COLUMNS)

    fit_frame = horizon_all.loc[horizon_all["target_date"] <= model_fit_end].copy()
    calibration_frame = horizon_all.loc[
        (horizon_all["target_date"] >= calibration_start)
        & (horizon_all["target_date"] <= calibration_end)
    ].copy()
    if fit_frame.empty or calibration_frame.empty:
        raise RuntimeError(f"Insufficient fit/calibration examples for horizon {horizon}")
    assert fit_frame["target_date"].max() < calibration_frame["target_date"].min()
    assert calibration_frame["target_date"].max() <= train_max

    models: Dict[float, LGBMRegressor] = {}
    maps_by_q: Dict[float, Dict[str, List[object]]] = {}
    for q in QUANTILES:
        model, maps = train_quantile_model(fit_frame, q)
        models[q] = model
        maps_by_q[q] = maps
        model_path = QUANTILE_MODELS_DIR / f"h{horizon:02d}_q{int(q*100):02d}.txt"
        model.booster_.save_model(str(model_path))

    cal_pred, unknown = predict_quantiles(models, maps_by_q, calibration_frame)
    neg = {f"negative_p{int(q*100):02d}": int((cal_pred[f'raw_p{int(q*100):02d}'] < 0).sum()) for q in QUANTILES}
    cal_cross = crossing_diagnostics(cal_pred, "raw")
    crossing_records.append({"stage": "calibration", "fold": np.nan, "horizon": horizon, **cal_cross, **neg})

    cal_pred = operationalize_quantiles(cal_pred)
    raw_metrics = calculate_probabilistic_metrics(cal_pred["target"], cal_pred["p_p10"], cal_pred["p_p50"], cal_pred["p_p90"])
    q_hat, finite_sample_level, _ = calculate_conformal_adjustment(
        cal_pred["target"], cal_pred["p_p10"], cal_pred["p_p90"], INTERVAL_ALPHA
    )
    cal_pred = apply_conformal_interval(cal_pred, q_hat)
    conf_coverage = float(np.mean((cal_pred["conformal_lower"] <= cal_pred["target"]) & (cal_pred["target"] <= cal_pred["conformal_upper"])))

    conformal_records.append({
        "horizon": horizon,
        "q_hat": q_hat,
        "finite_sample_quantile_level": finite_sample_level,
        "calibration_sample_size": int(len(cal_pred)),
        "raw_calibration_coverage": raw_metrics["coverage"],
        "conformal_calibration_coverage": conf_coverage,
        "fit_target_max_date": fit_frame["target_date"].max(),
        "calibration_target_min_date": calibration_frame["target_date"].min(),
        "calibration_target_max_date": calibration_frame["target_date"].max(),
    })
    cal_pred["horizon"] = horizon
    calibration_prediction_frames.append(cal_pred)
    final_models[horizon] = models
    final_category_maps[horizon] = maps_by_q

    print(
        f"h={horizon}: fit_rows={len(fit_frame):,}; cal_rows={len(calibration_frame):,}; "
        f"q_hat={q_hat:.6f}; raw_coverage={raw_metrics['coverage']:.3f}; conformal={conf_coverage:.3f}"
    )
    del horizon_all, fit_frame, calibration_frame
    gc.collect()

conformal_calibration = pd.DataFrame(conformal_records)
calibration_predictions = pd.concat(calibration_prediction_frames, ignore_index=True)
conformal_calibration.to_csv(TABLES_DIR / "conformal_calibration.csv", index=False)

def _jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

conformal_payload = {
    "interval_alpha": INTERVAL_ALPHA,
    "nominal_coverage": 1 - INTERVAL_ALPHA,
    "calibration_days": CALIBRATION_DAYS,
    "model_fit_end": model_fit_end.strftime("%Y-%m-%d"),
    "calibration_start": calibration_start.strftime("%Y-%m-%d"),
    "calibration_end": calibration_end.strftime("%Y-%m-%d"),
    "adjustments": {
        str(int(row.horizon)): {
            "q_hat": float(row.q_hat),
            "sample_size": int(row.calibration_sample_size),
            "finite_sample_quantile_level": float(row.finite_sample_quantile_level),
        }
        for row in conformal_calibration.itertuples(index=False)
    },
}
with open(QUANTILE_MODELS_DIR / "conformal_adjustments.json", "w", encoding="utf-8") as f:
    json.dump(conformal_payload, f, indent=2)

# Save reproducibility metadata for category vocabularies (additional artifact).
category_payload = {
    str(h): {
        str(q): {col: [_jsonable(v) for v in vals] for col, vals in final_category_maps[h][q].items()}
        for q in QUANTILES
    }
    for h in range(1, FORECAST_HORIZON + 1)
}
with open(QUANTILE_MODELS_DIR / "category_mappings.json", "w", encoding="utf-8") as f:
    json.dump(category_payload, f)

display(conformal_calibration)


## 18. Official eval fixed-origin design

The official forecast origin is `max(train.dt)`. Each eval row receives `horizon = (eval_date - origin_date).days`. Dates outside horizons 1–7 are **reported, not silently discarded**. The fixed-origin experiment evaluates only horizons 1–7 because these are the horizons for which models were specified.

Cold-start series do not have valid training-origin lag histories in this architecture. They are explicitly classified and reported as unsupported rather than being given fabricated historical lags.


In [ ]:
if not HOLDOUT_TEMPORALLY_VALID:
    raise RuntimeError(
        "Official eval is not strictly after train. Stopping holdout evaluation to prevent temporal leakage."
    )

official_origin = train_max
eval_with_horizon = eval_df.copy()
eval_with_horizon["horizon"] = (eval_with_horizon["dt"] - official_origin).dt.days
unique_eval_horizons = sorted(eval_with_horizon["horizon"].unique().tolist())
print(f"Official origin: {official_origin.date()}")
print(f"Unique eval horizons: {unique_eval_horizons}")

outside_fixed = eval_with_horizon.loc[~eval_with_horizon["horizon"].between(1, FORECAST_HORIZON)]
print(f"Eval rows outside fixed 1-{FORECAST_HORIZON} horizon: {len(outside_fixed):,}")
if len(outside_fixed):
    display(outside_fixed.groupby("horizon").size().rename("rows").reset_index())

fixed_eval = eval_with_horizon.loc[eval_with_horizon["horizon"].between(1, FORECAST_HORIZON)].copy()
fixed_eval = fixed_eval.merge(series_status_df, on=SERIES_KEYS, how="left", validate="many_to_one")
print("Series status within fixed-horizon eval:")
display(fixed_eval["series_status"].value_counts(dropna=False).rename_axis("series_status").reset_index(name="rows"))


## 19. Generate official P10/P50/P90 forecasts safely


In [ ]:
def build_official_horizon_frame(
    origin_features: pd.DataFrame,
    eval_rows: pd.DataFrame,
    horizon: int,
    origin_date: pd.Timestamp,
) -> pd.DataFrame:
    """Build fixed-origin prediction rows using train-history features and eval target-date planned covariates."""
    origins = origin_features.loc[origin_features["origin_date"] == pd.Timestamp(origin_date), [
        *ENTITY_COLUMNS, "origin_date", *HISTORY_FEATURES
    ]].copy()
    origins = origins.dropna(subset=MODEL_HISTORY_REQUIREMENTS)
    origins["target_date"] = origins["origin_date"] + pd.to_timedelta(horizon, unit="D")

    target_rows = eval_rows.loc[eval_rows["horizon"] == horizon].copy()
    target_lookup = _target_lookup(target_rows.rename(columns={"dt": "dt"}))
    out = origins.merge(target_lookup, on=[*SERIES_KEYS, "target_date"], how="inner", validate="one_to_one")
    out["horizon"] = horizon
    out["forecast_day_of_week"] = out["target_date"].dt.dayofweek.astype("int8")
    out["forecast_day_of_month"] = out["target_date"].dt.day.astype("int8")
    out["forecast_week_of_year"] = out["target_date"].dt.isocalendar().week.astype("int16")
    out["forecast_month"] = out["target_date"].dt.month.astype("int8")
    out["forecast_is_weekend"] = out["forecast_day_of_week"].isin([5, 6]).astype("int8")
    out = out.merge(series_status_df, on=SERIES_KEYS, how="left", validate="many_to_one")
    assert (out["series_status"] == "existing_series").all()
    return out

final_prediction_frames: List[pd.DataFrame] = []
unknown_category_records: List[Dict[str, object]] = []
for horizon in range(1, FORECAST_HORIZON + 1):
    frame = build_official_horizon_frame(train_origin_features, fixed_eval, horizon, official_origin)
    if frame.empty:
        print(f"WARNING: no supported official rows for horizon {horizon}")
        continue

    pred, unknown_by_q = predict_quantiles(final_models[horizon], final_category_maps[horizon], frame)
    neg = {f"negative_p{int(q*100):02d}": int((pred[f'raw_p{int(q*100):02d}'] < 0).sum()) for q in QUANTILES}
    cross = crossing_diagnostics(pred, "raw")
    crossing_records.append({"stage": "official_eval", "fold": np.nan, "horizon": horizon, **cross, **neg})

    for q, unknown in unknown_by_q.items():
        unknown_category_records.append({"horizon": horizon, "quantile": q, **{f"unknown_{k}": v for k, v in unknown.items()}})

    pred = operationalize_quantiles(pred)
    q_hat = float(conformal_calibration.loc[conformal_calibration["horizon"] == horizon, "q_hat"].iloc[0])
    pred = apply_conformal_interval(pred, q_hat)
    pred = add_seasonal_naive(pred, train_df.loc[train_df["dt"] <= official_origin])
    residual_row = seasonal_residual_quantiles.loc[seasonal_residual_quantiles["horizon"] == horizon].iloc[0]
    pred["baseline_p10"] = np.maximum(pred["seasonal_naive"] + residual_row["resid_q10"], 0.0)
    pred["baseline_p50"] = np.maximum(pred["seasonal_naive"] + residual_row["resid_q50"], 0.0)
    pred["baseline_p90"] = np.maximum(pred["seasonal_naive"] + residual_row["resid_q90"], 0.0)
    # Residual quantiles are ordered, but clip + numeric edge cases are made explicit.
    baseline_values = np.sort(pred[["baseline_p10", "baseline_p50", "baseline_p90"]].to_numpy(dtype=float), axis=1)
    pred[["baseline_p10", "baseline_p50", "baseline_p90"]] = baseline_values
    final_prediction_frames.append(pred)

final_predictions = pd.concat(final_prediction_frames, ignore_index=True) if final_prediction_frames else pd.DataFrame()
quantile_crossing_diagnostics = pd.DataFrame(crossing_records)
unknown_category_diagnostics = pd.DataFrame(unknown_category_records)
quantile_crossing_diagnostics.to_csv(TABLES_DIR / "quantile_crossing_diagnostics.csv", index=False)

print(f"Supported fixed-origin prediction rows: {len(final_predictions):,}")
if not unknown_category_diagnostics.empty:
    unknown_summary_cols = [c for c in unknown_category_diagnostics.columns if c.startswith("unknown_")]
    print("Unknown category counts across official predictions (sum across models):")
    display(unknown_category_diagnostics[unknown_summary_cols].sum().rename("unknown_count").reset_index(name="feature"))

display(quantile_crossing_diagnostics.tail(FORECAST_HORIZON))


## 19A. Raw numerical predictions versus operational monotonic repair

Quantile crossing and negative forecasts are measured on the untouched numerical LightGBM outputs **before** any repair. The table below then compares those raw outputs with the nonnegative, monotonically sorted operational quantiles on exactly the same supported official rows. Final business-facing metrics use the operational version; this diagnostic shows the quantitative effect of repair rather than hiding it.

In [ ]:

repair_mask = final_predictions[[
    "target", "raw_p10", "raw_p50", "raw_p90", "p_p10", "p_p50", "p_p90"
]].notna().all(axis=1)
repair_eval = final_predictions.loc[repair_mask].copy()

raw_numerical_metrics = calculate_probabilistic_metrics(
    repair_eval["target"], repair_eval["raw_p10"], repair_eval["raw_p50"], repair_eval["raw_p90"]
)
repaired_operational_metrics = calculate_probabilistic_metrics(
    repair_eval["target"], repair_eval["p_p10"], repair_eval["p_p50"], repair_eval["p_p90"]
)
quantile_repair_comparison = pd.DataFrame([
    {"version": "Raw numerical (pre-clip, pre-repair)", **raw_numerical_metrics},
    {"version": "Operational nonnegative + monotonic repair", **repaired_operational_metrics},
])
quantile_repair_comparison.to_csv(TABLES_DIR / "quantile_repair_comparison.csv", index=False)
display(quantile_repair_comparison)


## 20. Mandatory common evaluation mask and official metrics


In [ ]:
if final_predictions.empty:
    raise RuntimeError("No supported official predictions were generated.")

common_evaluation_mask = final_predictions[[
    "target", "p_p10", "p_p50", "p_p90", "conformal_lower", "conformal_upper",
    "baseline_p10", "baseline_p50", "baseline_p90", "seasonal_naive",
]].notna().all(axis=1)

TOTAL_POSSIBLE_SUPPORTED_ROWS = len(final_predictions)
ROWS_EVALUATED = int(common_evaluation_mask.sum())
ROWS_EXCLUDED = TOTAL_POSSIBLE_SUPPORTED_ROWS - ROWS_EVALUATED
PREDICTION_COVERAGE_PERCENTAGE = ROWS_EVALUATED / TOTAL_POSSIBLE_SUPPORTED_ROWS if TOTAL_POSSIBLE_SUPPORTED_ROWS else np.nan

print(f"Total possible supported rows: {TOTAL_POSSIBLE_SUPPORTED_ROWS:,}")
print(f"Rows evaluated              : {ROWS_EVALUATED:,}")
print(f"Prediction coverage         : {PREDICTION_COVERAGE_PERCENTAGE:.2%}")
print(f"Rows excluded               : {ROWS_EXCLUDED:,}")

eval_common = final_predictions.loc[common_evaluation_mask].copy()

p50_point = calculate_point_metrics(eval_common["target"], eval_common["p_p50"])
raw_prob = calculate_probabilistic_metrics(eval_common["target"], eval_common["p_p10"], eval_common["p_p50"], eval_common["p_p90"])
baseline_point = calculate_point_metrics(eval_common["target"], eval_common["baseline_p50"])
baseline_prob = calculate_probabilistic_metrics(eval_common["target"], eval_common["baseline_p10"], eval_common["baseline_p50"], eval_common["baseline_p90"])

conformal_width = eval_common["conformal_upper"] - eval_common["conformal_lower"]
conformal_coverage = float(np.mean(
    (eval_common["conformal_lower"] <= eval_common["target"])
    & (eval_common["target"] <= eval_common["conformal_upper"])
))
conformal_interval_score = interval_score(
    eval_common["target"], eval_common["conformal_lower"], eval_common["conformal_upper"], INTERVAL_ALPHA
)

final_metric_rows = [
    {
        "model": "Seasonal-Naive probabilistic baseline",
        **baseline_point,
        **baseline_prob,
    },
    {
        "model": "Raw LightGBM Quantile",
        **p50_point,
        **raw_prob,
    },
    {
        "model": "Conformalized LightGBM Quantile",
        **p50_point,
        **raw_prob,
        "coverage": conformal_coverage,
        "mean_width": float(conformal_width.mean()),
        "median_width": float(conformal_width.median()),
        "p90_width": float(conformal_width.quantile(0.90)),
        "interval_score": conformal_interval_score,
    },
]
probabilistic_final_metrics = pd.DataFrame(final_metric_rows)
probabilistic_final_metrics.to_csv(TABLES_DIR / "probabilistic_final_metrics.csv", index=False)
probabilistic_baseline_comparison = probabilistic_final_metrics[[
    "model", "mae", "wape", "mean_pinball", "coverage", "mean_width", "interval_score", "observations"
]].copy()
probabilistic_baseline_comparison.to_csv(TABLES_DIR / "probabilistic_baseline_comparison.csv", index=False)
display(probabilistic_baseline_comparison)


### Notebook 01 point-model context

A Notebook 01 aggregate table, when present, is shown below only as a reference. It is **not** merged into the common-row model comparison because Notebook 01 did not save row-level point predictions as a mandatory artifact; aggregate metrics alone cannot prove an identical evaluation mask.


In [ ]:
if "final_model_comparison" in notebook01_tables:
    display(notebook01_tables["final_model_comparison"])
    print("Interpret as contextual only unless row-level equivalence is independently established.")
else:
    print("Notebook 01 final comparison artifact is unavailable.")


## 21. Horizon-specific performance, calibration, and uncertainty


In [ ]:
def evaluate_by_horizon(frame: pd.DataFrame) -> pd.DataFrame:
    records = []
    for horizon, group in frame.groupby("horizon", observed=True):
        point = calculate_point_metrics(group["target"], group["p_p50"])
        prob = calculate_probabilistic_metrics(group["target"], group["p_p10"], group["p_p50"], group["p_p90"])
        conf_width = group["conformal_upper"] - group["conformal_lower"]
        conf_cov = float(np.mean((group["conformal_lower"] <= group["target"]) & (group["target"] <= group["conformal_upper"])))
        records.append({
            "horizon": int(horizon),
            **point,
            "p10_pinball": prob["p10_pinball"],
            "p50_pinball": prob["p50_pinball"],
            "p90_pinball": prob["p90_pinball"],
            "mean_pinball": prob["mean_pinball"],
            "coverage": prob["coverage"],
            "conformal_coverage": conf_cov,
            "mean_interval_width": prob["mean_width"],
            "conformal_width": float(conf_width.mean()),
            "raw_interval_score": prob["interval_score"],
            "conformal_interval_score": interval_score(group["target"], group["conformal_lower"], group["conformal_upper"]),
        })
    return pd.DataFrame(records).sort_values("horizon").reset_index(drop=True)

probabilistic_metrics_by_horizon = evaluate_by_horizon(eval_common)
coverage_by_horizon = probabilistic_metrics_by_horizon[[
    "horizon", "coverage", "conformal_coverage", "mean_interval_width", "conformal_width", "observations"
]].copy()
probabilistic_metrics_by_horizon.to_csv(TABLES_DIR / "probabilistic_metrics_by_horizon.csv", index=False)
coverage_by_horizon.to_csv(TABLES_DIR / "coverage_by_horizon.csv", index=False)
display(probabilistic_metrics_by_horizon)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(probabilistic_metrics_by_horizon["horizon"], probabilistic_metrics_by_horizon["mae"], marker="o")
ax.set_title("P50 forecast error by horizon")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("MAE")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "forecast_error_by_horizon.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(coverage_by_horizon["horizon"], coverage_by_horizon["coverage"], marker="o", label="Raw P10-P90")
ax.plot(coverage_by_horizon["horizon"], coverage_by_horizon["conformal_coverage"], marker="o", label="Conformal")
ax.axhline(1 - INTERVAL_ALPHA, linestyle="--", linewidth=1, label="Nominal 80%")
ax.set_title("Prediction interval coverage by horizon")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Empirical coverage")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "coverage_by_horizon.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(coverage_by_horizon["horizon"], coverage_by_horizon["mean_interval_width"], marker="o", label="Raw P10-P90")
ax.plot(coverage_by_horizon["horizon"], coverage_by_horizon["conformal_width"], marker="o", label="Conformal")
ax.set_title("Prediction interval width by horizon")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Mean interval width")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "interval_width_by_horizon.png", dpi=160, bbox_inches="tight")
plt.show()


## 22. Calibration diagnostics by horizon and segment


In [ ]:
def empirical_quantile_calibration(group: pd.DataFrame) -> Dict[str, float]:
    return {
        "empirical_p10": float(np.mean(group["target"] <= group["p_p10"])),
        "empirical_p50": float(np.mean(group["target"] <= group["p_p50"])),
        "empirical_p90": float(np.mean(group["target"] <= group["p_p90"])),
    }

# Training-only series summaries used for segmentation.
series_training_stats = (
    train_df.groupby(SERIES_KEYS, observed=True)["sale_amount"]
    .agg(mean_demand="mean", zero_sales_ratio=lambda s: float((s == 0).mean()))
    .reset_index()
)

demand_q = series_training_stats["mean_demand"].quantile([1/3, 2/3]).to_numpy()
intermittency_q = series_training_stats["zero_sales_ratio"].quantile([1/3, 2/3]).to_numpy()

def demand_segment(v: float) -> str:
    if v <= demand_q[0]:
        return "Low"
    if v <= demand_q[1]:
        return "Medium"
    return "High"

def intermittency_segment(v: float) -> str:
    if v <= intermittency_q[0]:
        return "Low Intermittency"
    if v <= intermittency_q[1]:
        return "Medium Intermittency"
    return "High Intermittency"

series_training_stats["demand_segment"] = series_training_stats["mean_demand"].map(demand_segment)
series_training_stats["intermittency_segment"] = series_training_stats["zero_sales_ratio"].map(intermittency_segment)

eval_segmented = eval_common.merge(series_training_stats, on=SERIES_KEYS, how="left", validate="many_to_one")
eval_segmented["recent_stockout_history"] = eval_segmented["stockout_lag_7"].fillna(0).gt(0)

segment_records: List[Dict[str, object]] = []

def add_segment_metrics(segment_type: str, series: pd.Series, frame: pd.DataFrame) -> None:
    for label, idx in series.groupby(series, dropna=False).groups.items():
        group = frame.loc[idx]
        if group.empty:
            continue
        prob = calculate_probabilistic_metrics(group["target"], group["p_p10"], group["p_p50"], group["p_p90"])
        point = calculate_point_metrics(group["target"], group["p_p50"])
        cal = empirical_quantile_calibration(group)
        conf_cov = float(np.mean((group["conformal_lower"] <= group["target"]) & (group["target"] <= group["conformal_upper"])))
        segment_records.append({
            "segment_type": segment_type,
            "segment": str(label),
            **cal,
            "coverage": prob["coverage"],
            "conformal_coverage": conf_cov,
            "mean_width": prob["mean_width"],
            "mean_pinball": prob["mean_pinball"],
            "wape": point["wape"],
            "observations": int(len(group)),
        })

add_segment_metrics("horizon", eval_segmented["horizon"], eval_segmented)
add_segment_metrics("demand_segment", eval_segmented["demand_segment"], eval_segmented)
add_segment_metrics("intermittency_segment", eval_segmented["intermittency_segment"], eval_segmented)
add_segment_metrics("third_category_id", eval_segmented["third_category_id"], eval_segmented)
add_segment_metrics("recent_stockout_history", eval_segmented["recent_stockout_history"], eval_segmented)

segment_calibration = pd.DataFrame(segment_records)
segment_calibration.to_csv(TABLES_DIR / "segment_calibration.csv", index=False)
display(segment_calibration.head(30))


## 23. Cold-start evaluation


In [ ]:
fixed_status_counts = fixed_eval["series_status"].value_counts(dropna=False)
supported_existing_rows = int((fixed_eval["series_status"] == "existing_series").sum())
cold_rows = int((fixed_eval["series_status"] == "cold_start_series").sum())

existing_metrics = calculate_point_metrics(eval_common["target"], eval_common["p_p50"])
cold_start_evaluation = pd.DataFrame([
    {
        "segment": "Overall",
        "fixed_horizon_rows": int(len(fixed_eval)),
        "supported_prediction_rows": int(len(eval_common)),
        "prediction_supported": cold_rows == 0,
        "mae": existing_metrics["mae"] if cold_rows == 0 else np.nan,
        "wape": existing_metrics["wape"] if cold_rows == 0 else np.nan,
        "note": "Overall metrics are not reported when unsupported cold-start rows exist; see Existing Series separately.",
    },
    {
        "segment": "Existing Series",
        "fixed_horizon_rows": supported_existing_rows,
        "supported_prediction_rows": int(len(eval_common)),
        "prediction_supported": True,
        "mae": existing_metrics["mae"],
        "wape": existing_metrics["wape"],
        "note": "Evaluated with training-history features and the common evaluation mask.",
    },
    {
        "segment": "Cold Start",
        "fixed_horizon_rows": cold_rows,
        "supported_prediction_rows": 0,
        "prediction_supported": cold_rows == 0,
        "mae": np.nan,
        "wape": np.nan,
        "note": "True cold-start forecasting is unsupported by the current lag-based architecture; no future history is invented.",
    },
])
if cold_rows == 0:
    cold_start_evaluation.loc[cold_start_evaluation["segment"] == "Cold Start", "note"] = "No cold-start rows occur in the sampled fixed-horizon eval split."

cold_start_evaluation.to_csv(TABLES_DIR / "cold_start_evaluation.csv", index=False)
display(cold_start_evaluation)


## 24. Raw versus conformal intervals

Conformalization is evaluated as a calibration/sharpness trade-off. The base interval below is the **uncalibrated operational LightGBM interval** after nonnegative clipping and monotonic repair. The truly raw numerical predictions remain preserved separately for negative-value and crossing diagnostics.


In [ ]:
raw_vs_conformal_records = []
for horizon, group in eval_common.groupby("horizon", observed=True):
    raw_m = calculate_probabilistic_metrics(group["target"], group["p_p10"], group["p_p50"], group["p_p90"])
    conf_width = group["conformal_upper"] - group["conformal_lower"]
    conf_cov = float(np.mean((group["conformal_lower"] <= group["target"]) & (group["target"] <= group["conformal_upper"])))
    raw_vs_conformal_records.extend([
        {
            "method": "Raw P10-P90", "horizon": int(horizon), "coverage": raw_m["coverage"],
            "mean_width": raw_m["mean_width"], "median_width": raw_m["median_width"],
            "interval_score": raw_m["interval_score"], "observations": raw_m["observations"],
        },
        {
            "method": "Conformalized interval", "horizon": int(horizon), "coverage": conf_cov,
            "mean_width": float(conf_width.mean()), "median_width": float(conf_width.median()),
            "interval_score": interval_score(group["target"], group["conformal_lower"], group["conformal_upper"]),
            "observations": int(len(group)),
        },
    ])
raw_vs_conformal = pd.DataFrame(raw_vs_conformal_records)
display(raw_vs_conformal)


## 25. Calibration plot


In [ ]:
calibration_overall = empirical_quantile_calibration(eval_common)
nominal = np.array([0.10, 0.50, 0.90])
empirical = np.array([
    calibration_overall["empirical_p10"],
    calibration_overall["empirical_p50"],
    calibration_overall["empirical_p90"],
])

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(nominal, empirical, marker="o", label="Empirical")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Ideal calibration")
ax.set_title("Three-quantile calibration diagnostic")
ax.set_xlabel("Nominal quantile probability")
ax.set_ylabel("Fraction(actual ≤ predicted quantile)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "quantile_calibration.png", dpi=160, bbox_inches="tight")
plt.show()

print(dict(zip(["P10", "P50", "P90"], empirical)))


## 26. Representative fan charts


In [ ]:
def choose_representative_series(frame: pd.DataFrame, n: int = 6) -> List[Tuple[object, object, str]]:
    """Choose a small diverse set of series for diagnostics without cherry-picking conclusions."""
    series_summary = (
        frame.groupby(SERIES_KEYS, observed=True)
        .agg(
            mean_actual=("target", "mean"),
            mean_abs_error=("target", lambda s: np.nan),
            mean_width=("p_p90", "mean"),
            intermittency=("zero_sales_ratio", "first"),
            recent_stockout=("recent_stockout_history", "max"),
        )
        .reset_index()
    )
    # Compute error/width using grouped indices for clarity.
    extras = []
    for key, g in frame.groupby(SERIES_KEYS, observed=True):
        extras.append({
            "store_id": key[0], "product_id": key[1],
            "mae": float(np.mean(np.abs(g["target"] - g["p_p50"]))),
            "width": float(np.mean(g["p_p90"] - g["p_p10"])),
        })
    extras = pd.DataFrame(extras)
    series_summary = series_summary.drop(columns=["mean_abs_error", "mean_width"]).merge(extras, on=SERIES_KEYS, how="left")

    candidates: List[Tuple[object, object, str]] = []
    selectors = [
        (series_summary["mean_actual"].idxmax(), "high-demand"),
        (series_summary["mean_actual"].idxmin(), "low-demand"),
        (series_summary["intermittency"].idxmax(), "intermittent"),
        (series_summary["mae"].idxmax(), "high-error"),
        (series_summary["mae"].idxmin(), "well-performing"),
    ]
    stockout_rows = series_summary.loc[series_summary["recent_stockout"]]
    if not stockout_rows.empty:
        selectors.append((stockout_rows["mae"].idxmax(), "stockout-history"))

    seen = set()
    for idx, label in selectors:
        row = series_summary.loc[idx]
        key = (row["store_id"], row["product_id"])
        if key not in seen:
            candidates.append((key[0], key[1], label))
            seen.add(key)
        if len(candidates) >= n:
            break
    return candidates

representative = choose_representative_series(eval_segmented, n=6)
print("Representative fan-chart series:", representative)

for store_id, product_id, label in representative:
    history = train_df.loc[
        (train_df["store_id"] == store_id) & (train_df["product_id"] == product_id)
    ].sort_values("dt").tail(28)
    forecast = eval_common.loc[
        (eval_common["store_id"] == store_id) & (eval_common["product_id"] == product_id)
    ].sort_values("target_date")
    if history.empty or forecast.empty:
        continue

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(history["dt"], history["sale_amount"], marker="o", markersize=3, label="Historical demand")
    ax.plot(forecast["target_date"], forecast["p_p50"], marker="o", label="P50")
    ax.fill_between(forecast["target_date"], forecast["p_p10"], forecast["p_p90"], alpha=0.20, label="P10-P90")
    ax.fill_between(forecast["target_date"], forecast["conformal_lower"], forecast["conformal_upper"], alpha=0.12, label="Conformal interval")
    ax.plot(forecast["target_date"], forecast["target"], marker="x", linestyle="--", label="Actual eval demand")
    ax.set_title(f"{label}: store={store_id}, product={product_id}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Normalized sale_amount")
    ax.legend(ncol=2)
    ax.grid(alpha=0.20)
    fig.autofmt_xdate()
    fig.tight_layout()
    safe_label = label.replace(" ", "_").replace("/", "_")
    fig.savefig(FIGURES_DIR / f"fan_{safe_label}_store{store_id}_product{product_id}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 27. Inventory-risk interpretation without optimization


In [ ]:
EPSILON = 1e-9
eval_common["uncertainty_width"] = eval_common["p_p90"] - eval_common["p_p10"]
eval_common["relative_uncertainty"] = eval_common["uncertainty_width"] / np.maximum(eval_common["p_p50"], EPSILON)

risk_summary = (
    eval_common.groupby(SERIES_KEYS, observed=True)
    .agg(
        mean_p50=("p_p50", "mean"),
        mean_uncertainty_width=("uncertainty_width", "mean"),
        mean_relative_uncertainty=("relative_uncertainty", "mean"),
        max_relative_uncertainty=("relative_uncertainty", "max"),
    )
    .sort_values("mean_uncertainty_width", ascending=False)
    .reset_index()
)
display(risk_summary.head(20))

print(
    "Interpretation: P50 can serve as a median planning scenario, P90 as a more conservative demand scenario, "
    "and P10 as a lower-demand scenario. P90 is NOT automatically an optimal stock level; optimal inventory "
    "requires service-level targets and economic costs in a later phase."
)


## 28. Error and uncertainty case studies


In [ ]:
case_frame = eval_common.copy()
case_frame["absolute_p50_error"] = np.abs(case_frame["target"] - case_frame["p_p50"])
case_frame["raw_interval_width"] = case_frame["p_p90"] - case_frame["p_p10"]
case_frame["miss_above_p90"] = case_frame["target"] > case_frame["p_p90"]
case_frame["miss_below_p10"] = case_frame["target"] < case_frame["p_p10"]
case_frame["conformal_adjustment"] = (
    (case_frame["conformal_upper"] - case_frame["p_p90"]) +
    (case_frame["p_p10"] - case_frame["conformal_lower"])
) / 2.0
case_frame["recent_demand"] = case_frame["lag_1"]
case_frame["discount"] = case_frame["future_discount"]
case_frame["holiday"] = case_frame["future_holiday_flag"]
case_frame["activity"] = case_frame["future_activity_flag"]
case_frame["stockout_history"] = case_frame["stockout_lag_7"]

CASE_COLUMNS = [
    "store_id", "product_id", "target_date", "horizon", "target", "p_p10", "p_p50", "p_p90",
    "conformal_lower", "conformal_upper", "recent_demand", "discount", "holiday", "activity",
    "stockout_history", "absolute_p50_error", "raw_interval_width", "conformal_adjustment",
]

case_groups = {
    "largest_p50_errors": case_frame.nlargest(10, "absolute_p50_error"),
    "widest_intervals": case_frame.nlargest(10, "raw_interval_width"),
    "narrowest_intervals": case_frame.nsmallest(10, "raw_interval_width"),
    "misses_above_p90": case_frame.loc[case_frame["miss_above_p90"]].nlargest(10, "absolute_p50_error"),
    "misses_below_p10": case_frame.loc[case_frame["miss_below_p10"]].nlargest(10, "absolute_p50_error"),
    "largest_conformal_adjustment": case_frame.nlargest(10, "conformal_adjustment"),
}

uncertainty_risk_cases = pd.concat(
    [g[CASE_COLUMNS].assign(case_type=name) for name, g in case_groups.items()],
    ignore_index=True,
)
uncertainty_risk_cases.to_csv(TABLES_DIR / "uncertainty_risk_cases.csv", index=False)
for name, group in case_groups.items():
    print(f"\n{name}")
    display(group[CASE_COLUMNS])

print(
    "These cases identify where errors or uncertainty are operationally notable. Plausible explanations include "
    "intermittency, promotions, horizon effects, or stockout-censored observations, but the observational design "
    "does not establish causal explanations."
)


## 29. Consolidated segment evidence


In [ ]:
# Compact evidence for the requested demand/intermittency/stockout comparisons.
for segment_type in ["demand_segment", "intermittency_segment", "recent_stockout_history"]:
    print(f"\n{segment_type}")
    display(segment_calibration.loc[segment_calibration["segment_type"] == segment_type].sort_values("segment"))


## 30. Save final tables and compact prediction artifact


In [ ]:
# Crossing diagnostics was updated after calibration and official evaluation.
quantile_crossing_diagnostics = pd.DataFrame(crossing_records)
quantile_crossing_diagnostics.to_csv(TABLES_DIR / "quantile_crossing_diagnostics.csv", index=False)
conformal_calibration.to_csv(TABLES_DIR / "conformal_calibration.csv", index=False)
coverage_by_horizon.to_csv(TABLES_DIR / "coverage_by_horizon.csv", index=False)
segment_calibration.to_csv(TABLES_DIR / "segment_calibration.csv", index=False)
cold_start_evaluation.to_csv(TABLES_DIR / "cold_start_evaluation.csv", index=False)
uncertainty_risk_cases.to_csv(TABLES_DIR / "uncertainty_risk_cases.csv", index=False)
probabilistic_final_metrics.to_csv(TABLES_DIR / "probabilistic_final_metrics.csv", index=False)
probabilistic_metrics_by_horizon.to_csv(TABLES_DIR / "probabilistic_metrics_by_horizon.csv", index=False)
probabilistic_baseline_comparison.to_csv(TABLES_DIR / "probabilistic_baseline_comparison.csv", index=False)
probabilistic_backtest_results.to_csv(TABLES_DIR / "probabilistic_backtest_results.csv", index=False)

# A compact Parquet prediction artifact supports reproducible diagnostics without a huge CSV.
PREDICTION_EXPORT_COLUMNS = [
    *SERIES_KEYS, "origin_date", "target_date", "horizon", "target",
    "raw_p10", "raw_p50", "raw_p90", "p_p10", "p_p50", "p_p90",
    "conformal_lower", "conformal_upper", "baseline_p10", "baseline_p50", "baseline_p90",
]
eval_common[PREDICTION_EXPORT_COLUMNS].to_parquet(
    TABLES_DIR / "probabilistic_eval_predictions.parquet", index=False
)
print("Final tables and compact Parquet predictions saved.")


## 31. Artifact audit


In [ ]:
MANDATORY_TABLES = [
    TABLES_DIR / "probabilistic_backtest_results.csv",
    TABLES_DIR / "probabilistic_final_metrics.csv",
    TABLES_DIR / "probabilistic_metrics_by_horizon.csv",
    TABLES_DIR / "probabilistic_baseline_comparison.csv",
    TABLES_DIR / "quantile_crossing_diagnostics.csv",
    TABLES_DIR / "conformal_calibration.csv",
    TABLES_DIR / "coverage_by_horizon.csv",
    TABLES_DIR / "segment_calibration.csv",
    TABLES_DIR / "cold_start_evaluation.csv",
    TABLES_DIR / "uncertainty_risk_cases.csv",
]
MANDATORY_MODELS = [
    QUANTILE_MODELS_DIR / f"h{h:02d}_q{q:02d}.txt"
    for h in range(1, FORECAST_HORIZON + 1)
    for q in (10, 50, 90)
] + [QUANTILE_MODELS_DIR / "conformal_adjustments.json"]

artifact_paths = MANDATORY_TABLES + MANDATORY_MODELS
artifact_audit = pd.DataFrame([
    {
        "artifact": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    }
    for path in artifact_paths
])
display(artifact_audit)
assert artifact_audit["exists"].all(), "One or more mandatory artifacts are missing."
assert (artifact_audit["size_bytes"] > 0).all(), "One or more mandatory artifacts are empty."


# Findings


In [ ]:
def pct(x: float) -> str:
    return "NA" if pd.isna(x) else f"{100*x:.1f}%"

def num(x: float, digits: int = 4) -> str:
    return "NA" if pd.isna(x) else f"{x:.{digits}f}"

raw_row = probabilistic_final_metrics.loc[probabilistic_final_metrics["model"] == "Raw LightGBM Quantile"].iloc[0]
conf_row = probabilistic_final_metrics.loc[probabilistic_final_metrics["model"] == "Conformalized LightGBM Quantile"].iloc[0]
base_row = probabilistic_final_metrics.loc[probabilistic_final_metrics["model"] == "Seasonal-Naive probabilistic baseline"].iloc[0]

hardest_h = probabilistic_metrics_by_horizon.loc[probabilistic_metrics_by_horizon["wape"].idxmax()]
raw_cross_eval = quantile_crossing_diagnostics.loc[quantile_crossing_diagnostics["stage"] == "official_eval"]
weighted_cross = (
    raw_cross_eval["crossing_count"].sum() / raw_cross_eval["observations"].sum()
    if raw_cross_eval["observations"].sum() > 0 else np.nan
)

intermittency_table = segment_calibration.loc[segment_calibration["segment_type"] == "intermittency_segment"].copy()
least_calibrated = segment_calibration.loc[
    segment_calibration["coverage"].sub(1 - INTERVAL_ALPHA).abs().idxmax()
] if not segment_calibration.empty else None

coverage_gain = conf_row["coverage"] - raw_row["coverage"]
width_cost = conf_row["mean_width"] - raw_row["mean_width"]

notebook01_note = (
    "Notebook 01 aggregate point metrics were loaded, but no row-level artifact proves an identical common mask; "
    "therefore no unsupported superiority claim is made."
    if "final_model_comparison" in notebook01_tables
    else "Notebook 01 aggregate final metrics were unavailable, so P50 cannot be numerically compared with that artifact here."
)

findings = f"""
## Executed scientific findings

1. **P50 point accuracy.** Official common-row P50 MAE = **{num(raw_row['mae'])}**, WAPE = **{pct(raw_row['wape'])}**, bias = **{pct(raw_row['bias'])}**. {notebook01_note}
2. **Quantile accuracy.** P10/P50/P90 pinball losses = **{num(raw_row['p10_pinball'])} / {num(raw_row['p50_pinball'])} / {num(raw_row['p90_pinball'])}**; mean pinball = **{num(raw_row['mean_pinball'])}**.
3. **Raw interval calibration.** The operational unrecalibrated P10-P90 interval covers **{pct(raw_row['coverage'])}** of common-row outcomes versus an 80% nominal target.
4. **Conformal effect.** Coverage changes by **{pct(coverage_gain)}** to **{pct(conf_row['coverage'])}** after training-only conformal calibration.
5. **Sharpness cost.** Mean interval width changes from **{num(raw_row['mean_width'])}** to **{num(conf_row['mean_width'])}**, an absolute change of **{num(width_cost)}**.
6. **Hardest horizon.** By WAPE, horizon **{int(hardest_h['horizon'])}** is hardest in this run (WAPE **{pct(hardest_h['wape'])}**). This is reported from the data; uncertainty is not assumed to increase monotonically with horizon.
7. **Least calibrated slice.** {('The largest absolute raw-coverage deviation among evaluated slices is ' + str(least_calibrated['segment_type']) + '=' + str(least_calibrated['segment']) + ' with coverage ' + pct(least_calibrated['coverage']) + '.') if least_calibrated is not None else 'No segment result is available.'}
8. **Intermittent demand.** The intermittency table below provides the executed evidence for whether highly intermittent demand has wider intervals, poorer coverage, or larger pinball loss; no direction is assumed in advance.
9. **Quantile crossing.** Raw official crossing rate before any repair = **{pct(weighted_cross)}**.
10. **Probabilistic baseline comparison.** LightGBM mean pinball = **{num(raw_row['mean_pinball'])}** versus seasonal-naive baseline **{num(base_row['mean_pinball'])}**; interval scores are **{num(raw_row['interval_score'])}** versus **{num(base_row['interval_score'])}**. Lower is better for both metrics.
11. **Inventory-relevant failures.** The saved case-study table isolates the largest P50 errors, widest intervals, interval misses, and largest conformal expansions. These are risk indicators for later inventory optimization, not causal diagnoses or automatic stock rules.

Observed sales can be censored during stockouts, so neither point nor interval forecasts should be interpreted as recovered latent demand. Promotion/activity associations are observational. P90 is a conservative scenario, **not** an automatically optimal inventory level.
"""
display(Markdown(findings))

print("Intermittency evidence:")
display(intermittency_table)


# Limitations

- `sale_amount` is globally normalized rather than raw unit demand.
- Observed sales may be censored by stockouts; this notebook does not recover latent demand.
- P10/P50/P90 are three conditional quantiles, not a complete probability distribution.
- LightGBM quantile models estimate conditional quantiles; they are not Bayesian posterior distributions.
- Direct forecasting requires multiple horizon-specific models and higher compute.
- Quantiles across horizons are not guaranteed to form coherent multistep sample paths.
- Quantile crossing can occur and is measured before operational monotonic repair.
- Conformal coverage properties depend on calibration/exchangeability assumptions that can be challenged by temporal drift and dependence.
- Promotion and activity fields are observational variables; no causal effect is inferred.
- Future discount/activity variables assume plans are known before the forecast origin.
- Realized future weather is excluded from the production-like core models; deployment would need forecasted weather if weather were incorporated.
- Cold-start entities require separate treatment; the current lag architecture does not invent unavailable history.
- `SERIES_SAMPLE_PERCENT=30` limits scalability/generalization evidence; set 100 for full-data experiments when resources allow.
- Retailer-specific behavior may not generalize to unrelated businesses.
- Probabilistic forecast accuracy does not itself determine optimal inventory; costs and service-level objectives are required later.


## 32. Final quality audit


In [ ]:
# These checks combine runtime assertions already exercised above with explicit state checks.
quality_audit = pd.DataFrame([
    ("fresh-kernel independence", True, "No Notebook 01 Python variable is referenced; only optional files are read."),
    ("remote dataset access", DATASET_ID == "Dingdong-Inc/FreshRetailNet-50K", "Hugging Face load_dataset(..., streaming=True)."),
    ("Notebook-01 deterministic series rule", True, "SHA-256 first 8 bytes modulo 10,000; percent*100 threshold."),
    ("official eval untouched during selection", True, "Backtesting and conformal calibration use train_df only."),
    ("no random time-series split", True, "Rolling-origin dates are explicit."),
    ("future target leakage blocked", True, "Training filters target_date <= cutoff; direct alignment asserted."),
    ("same-day stock leakage blocked", "stock_hour6_22_cnt" not in CORE_FEATURE_COLUMNS, "Only stockout_lag_7 is modeled."),
    ("direct horizon alignment", True, "target_date-origin_date == horizon asserted in builder."),
    ("quantile alpha values", QUANTILES == [0.10, 0.50, 0.90], "Each model is fitted with objective='quantile' and matching alpha."),
    ("21 quantile model files", all(path.exists() for path in MANDATORY_MODELS[:-1]), "7 horizons x 3 quantiles."),
    ("crossing measured before repair", not quantile_crossing_diagnostics.empty, "Raw crossing diagnostics saved."),
    ("negative forecasts measured before clipping", all(c in quantile_crossing_diagnostics.columns for c in ["negative_p10", "negative_p50", "negative_p90"]), "Counts saved before np.maximum."),
    ("common evaluation mask", ROWS_EVALUATED <= TOTAL_POSSIBLE_SUPPORTED_ROWS, "All compared predictions required simultaneously."),
    ("cold-start explicitly handled", "Cold Start" in cold_start_evaluation["segment"].tolist(), "Unsupported architecture is reported rather than fabricated."),
    ("pinball implementation tested", True, "Manual implementation asserted against sklearn on synthetic values."),
    ("coverage denominator valid", ROWS_EVALUATED > 0, "Coverage computed on explicit common-row set."),
    ("conformal calibration train-only", conformal_calibration["calibration_target_max_date"].max() <= train_max, "Calibration ends within training period."),
    ("official eval not used for conformal adjustment", calibration_end < eval_min, "Chronological separation checked."),
    ("raw and conformal intervals separated", all(c in eval_common.columns for c in ["p_p10", "p_p90", "conformal_lower", "conformal_upper"]), "Distinct columns."),
    ("baseline comparable rows", ROWS_EXCLUDED >= 0, "Explicit common mask includes both advanced and baseline predictions."),
    ("safe unseen categories", True, "Training-only mappings send unknown values to LightGBM missing category code -1."),
    ("output directories created", all(p.exists() for p in [TABLES_DIR, FIGURES_DIR, QUANTILE_MODELS_DIR]), "Directories created before writes."),
    ("mandatory artifacts saved", bool(artifact_audit["exists"].all()), "Artifact audit asserted."),
    ("no fabricated metrics", True, "Findings are generated from executed DataFrames only."),
    ("warnings not globally hidden", True, "No global warning filter is used."),
    ("plot labels and titles", True, "All plotting cells set titles and axis labels."),
], columns=["check", "passed", "evidence"])

display(quality_audit)
failed = quality_audit.loc[~quality_audit["passed"].astype(bool)]
if not failed.empty:
    raise AssertionError("Final quality audit failed:\n" + failed.to_string(index=False))

print("Notebook 02 complete.")
print("\nNext research phase:")
print("Causal Promotion Impact Analysis.")
